In [19]:
!pip install PySastrawi

import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [20]:
# Sesuaikan path sesuai lokasi file Anda di Colab
df_raw = pd.read_csv('dataset-abstrak.csv')
df = df_raw[['Kategori', 'Abstrak']].copy()
df.head()

,Kategori,Abstrak
0,RPL,Sistem informasi akademik (SIAKAD) merupaka...
1,RPL,Berjalannya koneksi jaringan komputer dengan l...
2,RPL,Web server adalah sebuah perangkat lunak serve...
3,KOMPUTASI,Penjadwalan kuliah di Perguruan Tinggi me...
4,RPL,Seiring perkembangan teknologi yang ada diduni...


In [21]:
def clean_text(text):
    text = str(text).lower() # Case folding
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"\s—\s", " ", text)
    text = re.sub(r"[,!\"#$%&()*+-.…/:;<=>?@[\]^_`{|}~\n]", " ", text) # Mengganti loop dengan regex yang lebih cepat
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['abstrak_clean'] = df['Abstrak'].apply(clean_text)
df[['Abstrak', 'abstrak_clean']].head()

,Abstrak,abstrak_clean
0,Sistem informasi akademik (SIAKAD) merupaka...,sistem informasi akademik siakad merupakan sis...
1,Berjalannya koneksi jaringan komputer dengan l...,berjalannya koneksi jaringan komputer dengan l...
2,Web server adalah sebuah perangkat lunak serve...,web server adalah sebuah perangkat lunak serve...
3,Penjadwalan kuliah di Perguruan Tinggi me...,penjadwalan kuliah di perguruan tinggi merupak...
4,Seiring perkembangan teknologi yang ada diduni...,seiring perkembangan teknologi yang ada diduni...


In [22]:
# Di kode lama, stopwords sudah diimpor tapi belum diaplikasikan ke data
stop_words = set(stopwords.words('indonesian'))

def remove_stopwords(text):
    tokens = text.split()
    filtered_tokens = [word for word in tokens if word not in stop_words]
    return " ".join(filtered_tokens)

df['abstrak_no_stopword'] = df['abstrak_clean'].apply(remove_stopwords)
df[['abstrak_clean', 'abstrak_no_stopword']].head()

,abstrak_clean,abstrak_no_stopword
0,sistem informasi akademik siakad merupakan sis...,sistem informasi akademik siakad sistem inform...
1,berjalannya koneksi jaringan komputer dengan l...,berjalannya koneksi jaringan komputer lancar g...
2,web server adalah sebuah perangkat lunak serve...,web server perangkat lunak server berfungsi me...
3,penjadwalan kuliah di perguruan tinggi merupak...,penjadwalan kuliah perguruan kompleks permasal...
4,seiring perkembangan teknologi yang ada diduni...,seiring perkembangan teknologi didunia muncul ...


In [23]:
factory = StemmerFactory()
stemmer = factory.create_stemmer()
term_dict = {}

# Menggunakan kamus (dictionary) agar kata yang sama tidak di-stem berulang kali
def stem_text(text):
    tokens = text.split()
    stemmed_tokens = []
    for term in tokens:
        if term not in term_dict:
            term_dict[term] = stemmer.stem(term)
        stemmed_tokens.append(term_dict[term])
    return " ".join(stemmed_tokens)

df['abstrak_stem'] = df['abstrak_no_stopword'].apply(stem_text)
df[['abstrak_no_stopword', 'abstrak_stem']].head()

,abstrak_no_stopword,abstrak_stem
0,sistem informasi akademik siakad sistem inform...,sistem informasi akademik siakad sistem inform...
1,berjalannya koneksi jaringan komputer lancar g...,jalan koneksi jaring komputer lancar ganggu ha...
2,web server perangkat lunak server berfungsi me...,web server perangkat lunak server fungsi terim...
3,penjadwalan kuliah perguruan kompleks permasal...,jadwal kuliah guru kompleks masalah variabel t...
4,seiring perkembangan teknologi didunia muncul ...,iring kembang teknologi dunia muncul teknologi...


In [24]:
X_text = df['abstrak_stem']
Y = df['Kategori'].values

# Mesin KNN hanya bisa membaca angka. Teks harus diubah menjadi bobot TF-IDF terlebih dahulu.
tfidf_vectorizer = TfidfVectorizer()
X_tfidf = tfidf_vectorizer.fit_transform(X_text)

print("Bentuk dimensi matriks TF-IDF:", X_tfidf.shape)

Bentuk dimensi matriks TF-IDF: (805, 6569)


In [25]:
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, Y, test_size=0.1, random_state=100
)
print("Jumlah data latih:", X_train.shape[0])
print("Jumlah data uji:", X_test.shape[0])

Jumlah data latih: 724
Jumlah data uji: 81


In [26]:
knn_asli = KNeighborsClassifier(n_neighbors=3)
knn_asli.fit(X_train, y_train)
y_pred_asli = knn_asli.predict(X_test)

print("Accuracy KNN (Tanpa PCA) :", accuracy_score(y_test, y_pred_asli))

Accuracy KNN (Tanpa PCA) : 0.8271604938271605


In [27]:
pca = PCA(n_components=30, random_state=42)

# PCA standar sklearn tidak menerima matriks sparse (sparse matrix) dari TF-IDF.
# Kita harus mengubahnya menjadi matriks padat menggunakan .toarray()
X_train_pca = pca.fit_transform(X_train.toarray())
X_test_pca = pca.transform(X_test.toarray())

print("Total Variance Explained:", sum(pca.explained_variance_ratio_))

Total Variance Explained: 0.22240109218259022


In [28]:
knn_pca = KNeighborsClassifier(n_neighbors=3)
knn_pca.fit(X_train_pca, y_train)
y_pred_pca = knn_pca.predict(X_test_pca)

print("Accuracy KNN (Dengan PCA) :", accuracy_score(y_test, y_pred_pca))

Accuracy KNN (Dengan PCA) : 0.8641975308641975
